# لاب K-Means — التعلم غير الخاضع للإشراف
لا نستخدم `churned_30d`. نبحث عن شرائح سلوكية ثم نفسرها بعد التدريب.

In [ ]:
import io
import pandas as pd
from google.colab import files
uploaded = files.upload()
name = next(iter(uploaded))
df = pd.read_csv(io.BytesIO(uploaded[name]))
features = ['orders_per_month','avg_basket_sar','days_since_last_order','promo_usage_rate']
X = df[features].copy()
display(X.describe().round(2))

## 1) عوّض المفقود وطبّق Scaling
التحجيم ضروري لأن K-Means يعتمد على المسافة.

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
prep = Pipeline([('impute', SimpleImputer(strategy='median')), ('scale', StandardScaler())])
Z = prep.fit_transform(X)
print('شكل البيانات بعد التجهيز:', Z.shape)

## 2) جرّب قيمًا مختلفة لـ K

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
rows = []
for k in range(2, 7):
    model = KMeans(n_clusters=k, n_init=10, random_state=42)
    labels = model.fit_predict(Z)
    rows.append({'K':k, 'Silhouette':silhouette_score(Z, labels), 'Inertia':model.inertia_})
display(pd.DataFrame(rows).round(3))

## 3) اختر K وفسّر المجموعات
لا تختَر أعلى رقم فقط؛ تأكد أن المجموعات مفهومة ويمكن استخدامها.

In [ ]:
chosen_k = 3
model = KMeans(n_clusters=chosen_k, n_init=10, random_state=42)
df['cluster'] = model.fit_predict(Z)
profile = df.groupby('cluster')[features].mean().round(2)
display(profile)

## اكتب تفسيرك
لكل مجموعة: اسم وصفي، ما الذي يميزها، واستخدام عملي واحد.

In [ ]:
cluster_notes = {0:{'الاسم':'','ما يميزها':'','الاستخدام':''},1:{'الاسم':'','ما يميزها':'','الاستخدام':''},2:{'الاسم':'','ما يميزها':'','الاستخدام':''}}
display(pd.DataFrame(cluster_notes).T)

## التسليم
سبب اختيار K، جدول `profile`، اسم وصفي لكل مجموعة، استخدام عملي واحد، وحد واحد يوضح لماذا العناقيد ليست حقيقة نهائية.